# 01 — Contrôler et préparer le catalogue Goodreads

Je travaille sur les **23 fichiers de livres de la version 18** du dataset Kaggle de Bahram Jannesar. Les données datent de 2020. Je n'utilise pas les fichiers de notes individuelles.

**Grain attendu : une fiche Goodreads identifiée par `Id`.** Une fiche n'est pas forcément une œuvre unique : plusieurs éditions peuvent coexister. Les compteurs de notes peuvent être partagés entre éditions ; je ne les additionnerai pas pour compter des lecteurs.

Objectif : garder une table fiable pour décrire le catalogue, puis préparer une sélection éditoriale en français. Les sources brutes restent intactes.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Le notebook fonctionne depuis son dossier ou depuis la racine du projet.
racine = Path.cwd()
if racine.name == 'notebooks':
    racine = racine.parent
dossier_sortie = racine / 'data/processed'
dossier_sortie.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 14)

## 1. Lire le même périmètre dans les 23 fichiers

Les fichiers n'ont pas tous les mêmes colonnes ni le même ordre. Je conserve les 18 colonnes communes utiles à l'audit. `PagesNumber` et `pagesNumber` désignent le même champ. Les descriptions et le second compteur de critiques, absents de certains fichiers, restent hors périmètre.

J'ajoute le nom du fichier et la position de l'enregistrement pour retrouver une anomalie. Ce numéro est une position dans les données lues, pas une ligne physique du CSV si du texte contient des retours à la ligne.

In [2]:
import json
import hashlib

manifest = json.loads((racine / 'data/sources.json').read_text(encoding='utf-8'))
colonnes = [
    'Id', 'Name', 'Authors', 'ISBN', 'Rating', 'PublishYear',
    'PublishMonth', 'PublishDay', 'Publisher', 'RatingDist1', 'RatingDist2',
    'RatingDist3', 'RatingDist4', 'RatingDist5', 'RatingDistTotal',
    'CountsOfReview', 'Language', 'pagesNumber'
]
tables = []
suivi_sources = []

for fichier in manifest['files']:
    chemin = racine / 'data/raw/kaggle_v18' / fichier['name']
    # Le contrôle de l'empreinte évite de mélanger des versions.
    assert hashlib.sha256(chemin.read_bytes()).hexdigest() == fichier['sha256']
    table = pd.read_csv(
        chemin, dtype='string',
        usecols=lambda nom: nom in colonnes or nom == 'PagesNumber'
    ).rename(columns={'PagesNumber': 'pagesNumber'})
    assert set(colonnes).issubset(table.columns)

    for colonne in ['Id', 'Rating', 'PublishYear', 'PublishMonth',
                    'PublishDay', 'pagesNumber', 'CountsOfReview']:
        table[colonne] = pd.to_numeric(table[colonne], errors='coerce')

    # Exemple : '5:123' devient 123, et 'total:456' devient 456.
    for colonne in ['RatingDist1', 'RatingDist2', 'RatingDist3',
                    'RatingDist4', 'RatingDist5', 'RatingDistTotal']:
        prefixe = 'total' if colonne == 'RatingDistTotal' else colonne[-1]
        nombre = table[colonne].str.extract('^' + prefixe + r':(\d+)$')[0]
        table[colonne] = pd.to_numeric(nombre, errors='coerce')

    table['FichierSource'] = fichier['name']
    table['LigneSource'] = np.arange(1, len(table) + 1)
    suivi_sources.append({'FichierSource': fichier['name'], 'LignesBrutes': len(table)})
    tables.append(table)

df_brut = pd.concat(tables, ignore_index=True)
del tables, table
nb_lignes_brutes = len(df_brut)
print('Fichiers :', len(suivi_sources))
print('Lignes brutes :', nb_lignes_brutes)
display(df_brut.head(3))

Fichiers : 23
Lignes brutes : 1850310


,Id,Name,RatingDist1,pagesNumber,RatingDist4,RatingDistTotal,PublishMonth,...,Rating,RatingDist2,RatingDist5,ISBN,RatingDist3,FichierSource,LigneSource
0,1,Harry Potter and the Half-Blood Prince (Harry ...,9896,652,556485,2298124,16,...,4.57,25317,1546466,<NA>,159960,book1-100k.csv,1
1,2,Harry Potter and the Order of the Phoenix (Har...,12455,870,604283,2358637,1,...,4.5,37005,1493113,0439358078,211781,book1-100k.csv,2
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,108202,309,1513191,6587388,1,...,4.47,130310,4268227,<NA>,567458,book1-100k.csv,3


## 2. Vérifier les manquants, les clés et les distributions

Une conversion impossible devient une valeur manquante. Le fichier brut permet de retrouver la valeur d'origine. Je regarde les extrêmes pour orienter le contrôle, sans supprimer automatiquement les grandes valeurs.

In [3]:
display(df_brut[colonnes].isna().sum().rename('ValeursManquantes').to_frame())
print('ID répétés après la première occurrence :', df_brut['Id'].duplicated().sum())
print('Répétitions sur les 18 colonnes auditées :', df_brut.duplicated(colonnes).sum())
display(df_brut[['Rating', 'RatingDistTotal', 'pagesNumber', 'PublishYear']]
        .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

,ValeursManquantes
Id,0
Name,0
Authors,0
ISBN,5923
Rating,0
PublishYear,0
PublishMonth,0
PublishDay,0
Publisher,17826
RatingDist1,0


ID répétés après la première occurrence : 195


Répétitions sur les 18 colonnes auditées : 112


,Rating,RatingDistTotal,pagesNumber,PublishYear
count,1850310.0,1850309.0,1850310.0,1850310.0
mean,2.89,4079.37,276.55,1997.84
std,1.73,71659.01,5006.02,87.89
min,0.0,0.0,0.0,1.0
25%,2.0,1.0,144.0,1993.0
50%,3.68,5.0,239.0,2000.0
75%,4.0,44.0,338.0,2005.0
95%,4.67,2841.0,603.0,2008.0
99%,5.0,51934.36,1002.0,2010.0
max,5.0,7094687.0,4517845.0,65535.0


## 3. Traiter les répétitions et mettre les conflits à part

Je retire les répétitions identiques sur les **18 colonnes auditées**, en conservant la première provenance dans l'ordre du manifeste. Je ne dis pas que les descriptions non lues sont identiques.

Si un même `Id` conserve plusieurs versions différentes, je mets **toutes** ces versions en quarantaine : sans date de collecte par ligne, je ne peux pas choisir la plus récente. Un identifiant invalide ou un titre absent empêche aussi de conserver la fiche dans la table principale.

In [4]:
nb_repetitions = int(df_brut.duplicated(colonnes).sum())
df_unique = df_brut.drop_duplicates(colonnes).copy()

id_invalide = df_unique['Id'].isna() | df_unique['Id'].le(0) | df_unique['Id'].mod(1).ne(0)
titre_absent = df_unique['Name'].fillna('').str.strip().eq('')
id_conflit = df_unique['Id'].duplicated(keep=False) & ~id_invalide

df_unique['MotifQuarantaine'] = np.select(
    [id_invalide.to_numpy(dtype=bool), id_conflit.to_numpy(dtype=bool), titre_absent.to_numpy(dtype=bool)],
    ['Identifiant invalide', 'Versions différentes pour un même Id', 'Titre absent'],
    default=''
)
quarantaine = df_unique.loc[df_unique['MotifQuarantaine'].ne('')].copy()
df = df_unique.loc[df_unique['MotifQuarantaine'].eq('')].drop(columns='MotifQuarantaine').copy()

assert df['Id'].is_unique
assert nb_lignes_brutes == nb_repetitions + len(quarantaine) + len(df)
print('Répétitions retirées :', nb_repetitions)
print('Lignes en quarantaine :', len(quarantaine))
print('Identifiants en quarantaine :', quarantaine['Id'].nunique())
print('Fiches conservées :', len(df))
display(quarantaine['MotifQuarantaine'].value_counts().to_frame('Lignes'))
del df_brut, df_unique

Répétitions retirées : 112
Lignes en quarantaine : 166
Identifiants en quarantaine : 83
Fiches conservées : 1850032


,Lignes
MotifQuarantaine,
Versions différentes pour un même Id,166


## 4. Contrôler les notes et leur dénominateur

Je vérifie que les cinq effectifs de notes retrouvent le total annoncé, puis que leur moyenne retrouve `Rating` à **0,01 point près**, pour tenir compte de l'arrondi.

Une note nulle avec zéro notation ne devient pas un mauvais score : la note exploitable reste vide. Une note incohérente reste également vide, mais la fiche et l'alerte sont conservées. Le total de notes est un compteur associé à la fiche, pas un nombre de lecteurs uniques à additionner entre fiches.

In [5]:
df = df.rename(columns={
    'Id': 'IdLivre', 'Name': 'Titre', 'Authors': 'Auteurs',
    'Publisher': 'Editeur', 'Language': 'CodeLangueSource',
    'Rating': 'NoteSource', 'RatingDistTotal': 'NbNotes',
    'pagesNumber': 'PagesSource', 'PublishYear': 'AnneeSource'
})
notes = ['RatingDist1', 'RatingDist2', 'RatingDist3', 'RatingDist4', 'RatingDist5']
df['DistributionValide'] = (
    df[notes].notna().all(axis=1)
    & df[notes].ge(0).all(axis=1)
    & df[notes].sum(axis=1).eq(df['NbNotes'])
)
df['NoteRecalculee'] = df[notes].mul([1, 2, 3, 4, 5]).sum(axis=1) / df['NbNotes'].where(df['NbNotes'].gt(0))
df['NoteExploitable'] = (
    df['DistributionValide']
    & df['NbNotes'].gt(0)
    & df['NoteSource'].between(1, 5)
    & df['NoteSource'].sub(df['NoteRecalculee']).abs().le(0.01)
).fillna(False)
df['Note'] = df['NoteSource'].where(df['NoteExploitable'])

print('Distributions incohérentes :', (~df['DistributionValide']).sum())
print('Fiches sans aucune notation :', df['NbNotes'].eq(0).sum())
print('Fiches avec note exploitable :', df['NoteExploitable'].sum())
print('Notes non exploitables malgré un compteur positif :',
      (df['NbNotes'].gt(0) & ~df['NoteExploitable']).sum())

Distributions incohérentes : 1
Fiches sans aucune notation : 451777
Fiches avec note exploitable : 1398254
Notes non exploitables malgré un compteur positif : 0


## 5. Préparer les informations du catalogue

Je regroupe seulement les variantes évidentes de code langue. Une langue absente reste **Non renseignée** ; je ne la déduis ni du titre ni du pays supposé de l'auteur.

Les colonnes `PublishMonth` et `PublishDay` semblent inversées dans la source. Je n'invente pas une date complète : j'utilise seulement les années entières entre 1 et 2020, date limite du jeu étudié. C'est une année de publication renseignée sur la fiche, pas une date de vente ou de notation.

Zéro page signifie ici une pagination inexploitable. Les grandes paginations restent présentes, car il peut s'agir de recueils. Aucun genre exploitable n'est fourni : je n'en crée pas.

In [6]:
for colonne in ['Titre', 'Auteurs', 'Editeur', 'ISBN', 'CodeLangueSource']:
    df[colonne] = df[colonne].astype('string').str.strip().replace('', pd.NA)

df['LangueAbsente'] = df['CodeLangueSource'].isna()
df['EditeurAbsent'] = df['Editeur'].isna()
df['AuteurAbsent'] = df['Auteurs'].isna()
df['CodeLangue'] = df['CodeLangueSource'].str.lower().replace({
    'en-us': 'eng', 'en-gb': 'eng', 'en-ca': 'eng', 'en': 'eng',
    'fra': 'fre', 'fr': 'fre'
}).fillna('non_renseigne')
langues = {'eng': 'Anglais', 'fre': 'Français', 'spa': 'Espagnol',
           'ger': 'Allemand', 'deu': 'Allemand', 'ita': 'Italien',
           'jpn': 'Japonais', 'por': 'Portugais', 'non_renseigne': 'Non renseignée'}
df['Langue'] = df['CodeLangue'].map(langues).fillna('Autre langue')

df['Pages'] = df['PagesSource'].where(df['PagesSource'].gt(0) & df['PagesSource'].mod(1).eq(0))
df['PagesAbsentes'] = df['Pages'].isna()
df['PagesAtypiques'] = df['Pages'].gt(5000).fillna(False)
df['AnneePublication'] = df['AnneeSource'].where(df['AnneeSource'].between(1, 2020) & df['AnneeSource'].mod(1).eq(0))
df['AnneeInvalide'] = df['AnneePublication'].isna()
df['Editeur'] = df['Editeur'].fillna('Non renseigné')
df['Auteurs'] = df['Auteurs'].fillna('Non renseigné')

df['TranchePages'] = pd.cut(df['Pages'], bins=[0, 150, 300, 600, np.inf],
                          labels=['1-150', '151-300', '301-600', '601 et plus']).astype('string').fillna('Non renseigné')
df['TrancheNotes'] = pd.cut(df['NbNotes'], bins=[-1, 0, 99, 999, 9999, np.inf],
                          labels=['0 note', '1-99', '100-999', '1 000-9 999', '10 000 et plus']).astype('string').fillna('Non renseigné')
df['PeriodePublication'] = pd.cut(df['AnneePublication'], bins=[0, 1949, 1979, 1999, 2009, 2020],
                                labels=['Avant 1950', '1950-1979', '1980-1999', '2000-2009', '2010-2020']).astype('string').fillna('Non renseignée')

# Un même titre et auteur peut cacher plusieurs éditions. Je signale, sans fusionner.
titre_normalise = (df['Titre'].str.casefold()
                   .str.replace(r'[^\w\s]', ' ', regex=True)
                   .str.replace(r'\s+', ' ', regex=True).str.strip())
auteur_normalise = df['Auteurs'].str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()
df['CleTitreAuteur'] = titre_normalise + ' | ' + auteur_normalise
df['TitreAuteurRepete'] = df['CleTitreAuteur'].duplicated(keep=False)
print('Langues absentes :', df['LangueAbsente'].sum())
print('Paginations non exploitables :', df['PagesAbsentes'].sum())
print('Années non exploitables :', df['AnneeInvalide'].sum())
print('Paginations au-delà de 5 000 pages, à vérifier :', df['PagesAtypiques'].sum())
print('Fiches partageant un titre et auteur normalisés :', df['TitreAuteurRepete'].sum())

Langues absentes : 1598339
Paginations non exploitables : 11172
Années non exploitables : 72
Paginations au-delà de 5 000 pages, à vérifier : 328
Fiches partageant un titre et auteur normalisés : 349270


## 6. Exporter avec les contrôles

La table principale conserve les notes sources, les compteurs, la provenance et les indicateurs de qualité. La quarantaine reste séparée. Les effectifs de motifs de qualité se chevauchent : ils ne doivent pas être additionnés pour obtenir un total de fiches.

In [7]:
controle_qualite = pd.DataFrame({
    'Controle': ['Lignes brutes', 'Répétitions retirées', 'Lignes en quarantaine',
                 'Fiches conservées', 'Langue absente', 'Editeur absent',
                 'Pagination inexploitable', 'Année inexploitable',
                 'Sans notation', 'Distribution incohérente',
                 'Note non exploitable avec compteur positif', 'Titre et auteur répétés',
                 'Pagination supérieure à 5 000 pages'],
    'Effectif': [nb_lignes_brutes, nb_repetitions, len(quarantaine), len(df),
                df['LangueAbsente'].sum(), df['EditeurAbsent'].sum(),
                df['PagesAbsentes'].sum(), df['AnneeInvalide'].sum(),
                df['NbNotes'].eq(0).sum(), (~df['DistributionValide']).sum(),
                (df['NbNotes'].gt(0) & ~df['NoteExploitable']).sum(), df['TitreAuteurRepete'].sum(),
                df['PagesAtypiques'].sum()]
})
assert df['IdLivre'].is_unique
assert not df['IdLivre'].isin(quarantaine['Id']).any()
assert df.loc[~df['NoteExploitable'], 'Note'].isna().all()
assert df['Note'].dropna().between(1, 5).all()

df.to_csv(dossier_sortie / 'livres.csv', index=False, sep=';', encoding='utf-8-sig')
quarantaine.to_csv(dossier_sortie / 'quarantaine.csv', index=False, sep=';', encoding='utf-8-sig')
controle_qualite.to_csv(dossier_sortie / 'controle_qualite.csv', index=False, sep=';', encoding='utf-8-sig')
pd.DataFrame(suivi_sources).to_csv(dossier_sortie / 'audit_sources.csv', index=False, sep=';', encoding='utf-8-sig')
display(controle_qualite)
print('Contrôles qualité et exports : OK')

,Controle,Effectif
0,Lignes brutes,1850310
1,Répétitions retirées,112
2,Lignes en quarantaine,166
3,Fiches conservées,1850032
4,Langue absente,1598339
5,Editeur absent,17819
6,Pagination inexploitable,11172
7,Année inexploitable,72
8,Sans notation,451777
9,Distribution incohérente,1


Contrôles qualité et exports : OK


## Ce que je garde pour la suite

Les données ne décrivent ni les ventes ni un historique d'activité. Une bonne note avec très peu de votes demande de la prudence. Je lirai chaque moyenne avec son effectif et la couverture des métadonnées.

La prochaine étape est [l'analyse globale](02_analyse_globale.ipynb). Les décisions et résultats sont repris dans [Qualité.md](../docs/Qualité.md).

**Résultat :** je conserve 1 850 032 fiches après 112 répétitions retirées et 166 lignes en quarantaine. La langue manque pour 86,40 % des fiches. Le compteur négatif de la fiche 3485352 (`total:-2` et `5:-2`) reste non exploitable ; il ne devient pas zéro. Ces limites orientent l'analyse suivante.